In [ ]:
# =============================================================
# SISTEMA DE VALIDACIÓN AUTOMATIZADA DE DOCUMENTOS ACADÉMICOS
# Pipeline multinivel (5 barreras): Clasificación de Actas y CURPs
# Referencia metodológica: Bulatov et al. (2021), MIDV-2020 (arXiv:2107.00396)
# =============================================================
# CONFIGURADO PARA: GOOGLE COLAB + GOOGLE DRIVE
# Ruta del proyecto en Drive:
#   Mi unidad → Proyecto de documentos → MiDataset
# =============================================================

# PASO 1: Montar Google Drive (solicita permiso la primera vez)
from google.colab import drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore')  # Supresión de advertencias para depuración visual

import pandas as pd              # Estructuras de datos tabulares
from pathlib import Path         # Gestión portable de rutas de archivos

# PASO 2: Ruta exacta al proyecto en Google Drive
# Estructura: Mi unidad/Proyecto de documentos/MiDataset
BASE_DIR = Path('/content/drive/MyDrive/Proyecto de documentos/MiDataset')

# PASO 3: Verificar que la ruta existe correctamente
print('Ruta configurada:', BASE_DIR)
print('¿Existe en Drive?:', BASE_DIR.exists())

if not BASE_DIR.exists():
    print('\n RUTA NO ENCONTRADA. Verifica que el nombre de la carpeta sea exacto.')
else:
    print('\n Conexión con Drive establecida correctamente.')

# PASO 4: Lectura del manifiesto de metadatos del dataset
df = pd.read_csv(BASE_DIR / 'manifiesto.csv')

# Vista preliminar de registros cargados
df.head()

In [2]:
# Análisis exploratorio preliminar de la consistencia del dataset

print('Registros totales:', df.shape[0])  # Cardinalidad del conjunto de datos
print('Columnas:        ', df.shape[1])   # Dimensionalidad de metadatos
print()

# Validación de integridad: detección de valores nulos (NaN) en metadatos
print('Valores nulos por columna:')
print(df.isnull().sum())
print()

# Distribución estadística y balanceo de clases en la muestra
df['clase'].value_counts()

Registros totales: 233
Columnas:         13

Valores nulos por columna:
nombre_archivo_original      0
id_documento                 0
id_pagina                    0
clase                        0
ruta_imagen                  0
ruta_pdf_original          205
ancho_pixeles              205
alto_pixeles               205
puntuacion_calidad           0
necesita_revision            0
es_aumentada                 0
fecha_procesado              0
notas                       28
dtype: int64



clase
curp                176
actas_nacimiento     57
Name: count, dtype: int64

In [ ]:
# =============================================================
# BARRERA 1: Clasificación de tipo de documento mediante CNN
# Objetivo: Identificar la categoría de documento (Acta o CURP)
# Metodología: Transfer Learning sobre arquitectura MobileNetV3-Small
# =============================================================

# Librerías para modelado profundo y métricas estadísticas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ── PASO 1: PREPROCESAMIENTO Y PARTICIÓN DEL DATASET ──────────
# Mapeo numérico de etiquetas nominales
CLASES     = {'actas_nacimiento': 0, 'curp': 1, 'otros': 2}
CLASES_INV = {v: k for k, v in CLASES.items()}

# Filtrado de consistencia física de imágenes existentes en almacenamiento
df_v = df[df['clase'].isin(CLASES)].copy()
df_v = df_v[df_v['ruta_imagen'].apply(lambda r: (BASE_DIR / r).exists())]

# Extracción de vectores de características y etiquetas objetivo
X = [str(BASE_DIR / r) for r in df_v['ruta_imagen']]
Y = [CLASES[c] for c in df_v['clase']]

# Partición estratificada (80% entrenamiento, 20% prueba)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)
print(f'Total: {len(X)} imágenes | Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}')

# ── PASO 2: PIPELINE DE TRANSFORMACIONES Y DATA AUGMENTATION ──
# Transformaciones para conjunto de entrenamiento (Aumentación de datos y regularización fotométrica)
tf_train = transforms.Compose([
    transforms.Resize((224, 224)),           # Escalado estándar para MobileNetV3
    transforms.RandomHorizontalFlip(p=0.3),  # Regularización por simetría horizontal
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Variaciones controladas de exposición
    transforms.ToTensor(),                   # Conversión a tensor [0.0, 1.0]
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Normalización estadística de ImageNet
])

# Transformaciones para conjunto de validación/prueba (Métricas sin alteración fotométrica)
tf_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Clase personalizada para carga bajo demanda (Lazy Loading) de imágenes
class DocumentDataset(Dataset):
    def __init__(self, rutas, etiquetas, transform):
        self.rutas     = rutas
        self.etiquetas = etiquetas
        self.transform = transform

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert('RGB')
        imagen_lista = self.transform(img)
        etiqueta = torch.tensor(self.etiquetas[idx], dtype=torch.long)
        return imagen_lista, etiqueta

# Definición de cargadores de datos con procesamiento por lotes (Minibatch size = 8)
loader_train = DataLoader(DocumentDataset(X_train, Y_train, tf_train), batch_size=8, shuffle=True)
loader_test  = DataLoader(DocumentDataset(X_test, Y_test, tf_test),  batch_size=8, shuffle=False)
print(f'Lotes de entrenamiento: {len(loader_train)} | Lotes de prueba: {len(loader_test)}')

# ── PASO 3: ARQUITECTURA DE LA RED NEURONAL (TRANSFER LEARNING) ──
# Detección del acelerador de cómputo disponible
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de entrenamiento: {dispositivo}')

# Carga de arquitectura MobileNetV3-Small con pesos pre-entrenados en ImageNet
modelo = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Congelación de parámetros en el extractor de características (Fase 2)
for p in modelo.features.parameters():
    p.requires_grad = False

# Redefinición del clasificador final (Fase 3: Multi-Layer Perceptron personalizado)
modelo.classifier = nn.Sequential(
    nn.Linear(576, 256),  # Capa lineal reductora de dimensionalidad
    nn.Hardswish(),        # Función de activación no lineal optimizada
    nn.Dropout(p=0.3),    # Capa de regularización para mitigar overfitting (30%)
    nn.Linear(256, 3)  # 3 clases: actas_nacimiento, curp, otros     # Capa lineal de salida para dos clases lógicas
)

modelo = modelo.to(dispositivo)

# ── PASO 4: OPTIMIZACIÓN Y BUCLE DE ENTRENAMIENTO ────────────────
criterio = nn.CrossEntropyLoss() # Función de pérdida de entropía cruzada

# Optimizador Adam aplicado exclusivamente a parámetros activos (clasificador)
optimizador = optim.Adam(filter(lambda p: p.requires_grad, modelo.parameters()), lr=0.001)

# Scheduler para decrecimiento adaptativo del learning rate ante estancamiento
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='max', patience=5, factor=0.5)

mejor_prec  = 0.0
sin_mejora  = 0
mejor_pesos = None

print(f"\n{'Época':<8} {'Loss':<12} {'Precisión':<12} {'Estado'}")
print('-' * 45)

# Bucle iterativo de optimización de pesos y validación
for epoca in range(40):
    # Fase de optimización en conjunto de entrenamiento
    modelo.train()
    loss_total = 0.0
    for imgs, lbls in loader_train:
        imgs, lbls = imgs.to(dispositivo), lbls.to(dispositivo)
        optimizador.zero_grad()
        salida = modelo(imgs)
        loss = criterio(salida, lbls)
        loss.backward()  # Cálculo automático de gradientes
        optimizador.step() # Actualización de parámetros
        loss_total += loss.item()

    # Fase de validación determinística en conjunto de prueba
    modelo.eval()
    correctas, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader_test:
            salida = modelo(imgs.to(dispositivo))
            _, pred = torch.max(salida, 1)
            total     += lbls.size(0)
            correctas += (pred.cpu() == lbls).sum().item()

    prec = correctas / total
    scheduler.step(prec)

    # Preservación de la mejor configuración de pesos hallada
    if prec > mejor_prec:
        mejor_prec  = prec
        sin_mejora  = 0
        mejor_pesos = {k: v.clone() for k, v in modelo.state_dict().items()}
        estado = '<- mejor'
    else:
        sin_mejora += 1
        estado = ''

    print(f'{epoca+1:<8} {loss_total/len(loader_train):<12.4f} {prec*100:<11.1f}% {estado}')

    # Early Stopping: Detiene el entrenamiento si no se registra mejoría en 15 épocas
    if sin_mejora >= 15:
        print(f'Early stopping en época {epoca+1}: sin mejora por {sin_mejora} épocas')
        break

# Restauración del modelo con la mejor configuración de pesos
modelo.load_state_dict(mejor_pesos)
modelo.eval()
print(f'\nMejor precisión obtenida en conjunto de prueba: {mejor_prec*100:.1f}%')

# Evaluación de rendimiento mediante métricas estándar
preds_all, lbls_all = [], []
with torch.no_grad():
    for imgs, lbls in loader_test:
        _, pred = torch.max(modelo(imgs.to(dispositivo)), 1)
        preds_all.extend(pred.cpu().numpy())
        lbls_all.extend(lbls.numpy())
print(classification_report(lbls_all, preds_all, target_names=list(CLASES.keys())))

# Matriz de confusión para análisis de falsos positivos/negativos
pd.DataFrame(
    confusion_matrix(lbls_all, preds_all),
    index=[f'Real: {n}' for n in CLASES],
    columns=[f'Pred: {n}' for n in CLASES]
)
# ── Persistencia del modelo entrenado ─────────────────────────
# Serialización de los pesos del modelo en formato PyTorch (.pth)
# para habilitar inferencia posterior sin reentrenamiento.
# Ruta: MiDataset/modelo/modelo_barrera1.pth
ruta_guardado = BASE_DIR / 'modelo' / 'modelo_barrera1.pth'
torch.save(modelo.state_dict(), ruta_guardado)
print(f'Modelo serializado: {ruta_guardado}')
print(f'Tamanio: {ruta_guardado.stat().st_size / 1024:.1f} KB')

In [ ]:
# ── PASO 5: EVALUACIÓN DE BARRERA 1 SOBRE EL DATASET COMPLETO ───
# Inferencia y filtro con umbral de decisión determinístico del 70%

def barrera_1(ruta):
    # Preparación de la imagen de entrada para inferencia
    tensor = tf_test(Image.open(ruta).convert('RGB')).unsqueeze(0).to(dispositivo)

    # Obtención de probabilidades lógicas de clase (probabilidades Softmax)
    with torch.no_grad():
        probs = torch.softmax(modelo(tensor), dim=1)[0]

    # Selección de la clase con mayor confianza
    idx  = int(probs.argmax())
    conf = float(probs.max())
    clase = CLASES_INV[idx]

    # Filtrado estricto por umbral de confianza mínimo
    if conf < 0.70:
        return False, clase, conf, (
            f'BARRERA 1 FALLIDA: tipo de documento no reconocido '
            f'({conf*100:.1f}% de confianza, mínimo requerido: 70%). '
            f'Sube una imagen más clara y bien encuadrada.'
        )
    return True, clase, conf, f"Documento reconocido como '{clase}' con {conf*100:.1f}% de confianza"

# Evaluación iterativa de todos los registros del dataset en Barrera 1
b1_res = []
for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    ok, clase_pred, conf, msg = barrera_1(ruta)
    b1_res.append({
        'archivo'    : fila.get('nombre_archivo_original', ruta.name),
        'clase_real' : fila['clase'],
        'clase_pred' : clase_pred,
        'confianza_%': round(conf*100, 1),
        'b1_ok'      : ok,
        'mensaje'    : msg
    })

B1 = pd.DataFrame(b1_res)

print('=== BARRERA 1 COMPLETADA ===')
print(f'Procesados: {len(B1)} archivos. Resultados listos para la Tabla Maestra.\n')


In [ ]:
# =============================================================
# CELDA 4: BARRERA 2 - VERIFICACIÓN DE CALIDAD DE IMAGEN
# =============================================================
# OBJETIVO: Rechazar documentos cuya imagen sea ilegible debido a:
#   1. DESENFOQUE (blur): imagen borrosa, movida o de baja resolución.
#   2. RUIDO EXCESIVO: mucho "grano" que impide leer el texto.
#   3. ROTACIÓN EXCESIVA (skew): documento muy torcido (> 15 grados).
#
# TÉCNICA: Operadores de visión por computadora (OpenCV)
#   - Varianza del Laplaciano: mide la nitidez.
#     Un valor bajo -> imagen borrosa.
#     Un valor alto -> imagen nítida.
#   - Varianza global: mide el contraste/ruido.
#   - Transformada de Hough probabilística: detecta líneas rectas
#     para estimar el ángulo de rotación del documento.
#
# UMBRALES (ajustables):
#   - Nitidez mínima (blur_threshold): 80.0
#   - Ruido mínimo  (noise_threshold): 10.0
#   - Skew máximo   (skew_threshold):  15.0 grados

import cv2
import numpy as np

BLUR_THRESHOLD  = 80.0
NOISE_THRESHOLD = 10.0
SKEW_THRESHOLD  = 15.0

def barrera_2(ruta_imagen):
    # Cargar imagen en escala de grises
    img_gray = cv2.imread(str(ruta_imagen), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        return False, 'Error: No se pudo leer la imagen.'

    # PRUEBA 1: DESENFOQUE
    varianza_blur = cv2.Laplacian(img_gray, cv2.CV_64F).var()
    if varianza_blur < BLUR_THRESHOLD:
        return False, f'Error: Imagen borrosa (nitidez={varianza_blur:.1f} < {BLUR_THRESHOLD})'

    # PRUEBA 2: RUIDO EXCESIVO
    varianza_ruido = np.var(img_gray)
    if varianza_ruido < NOISE_THRESHOLD:
        return False, f'Error: Contraste insuficiente (var={varianza_ruido:.1f})'

    # PRUEBA 3: ROTACIÓN (SKEW)
    bordes = cv2.Canny(img_gray, 50, 150, apertureSize=3)
    lineas = cv2.HoughLinesP(bordes, 1, np.pi/180,
                              threshold=100, minLineLength=100, maxLineGap=10)
    if lineas is not None:
        angulos = []
        for linea in lineas:
            x1, y1, x2, y2 = linea[0]
            if x2 != x1:
                angulo = np.degrees(np.arctan2(y2-y1, x2-x1))
                angulos.append(angulo)
        if angulos:
            skew = abs(np.median(angulos))
            if skew > SKEW_THRESHOLD:
                return False, f'Error: Documento muy rotado (skew={skew:.1f}° > {SKEW_THRESHOLD}°)'

    return True, f'Calidad OK (nitidez={varianza_blur:.1f})'

b2_res = []
aprobados_b1 = set(B1[B1['b1_ok']]['archivo']) if 'B1' in globals() else set()

# Procesar en silencio (sin prints adentro del ciclo)
for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    if nombre not in aprobados_b1: continue

    ok, msg = barrera_2(ruta)
    b2_res.append({'archivo': nombre, 'b2_ok': ok, 'mensaje': msg})

B2 = pd.DataFrame(b2_res)
print('=== BARRERA 2 COMPLETADA ===')
print(f'Procesados: {len(B2)} archivos. Resultados listos para la Tabla Maestra.\n')



In [ ]:
# =============================================================
# CELDA 5: BARRERA 3 - EXTRACCION DE INFORMACION CLAVE (KIE) Y OCR
# =============================================================
# OBJETIVO: Leer el texto del documento y verificar que los datos
#           clave estan presentes y son estructuralmente validos.
#
# POR QUE ES NECESARIA ESTA BARRERA?
#   Las Barreras 1 y 2 solo miran la forma y calidad de la imagen.
#   La Barrera 3 mira el CONTENIDO: un documento puede verse como
#   un CURP y estar en buena calidad, pero si el codigo no tiene
#   la estructura correcta de 18 caracteres, debe rechazarse.
#
# TECNOLOGIA USADA: EasyOCR
#   Motor de reconocimiento optico de caracteres basado en Deep Learning.
#   Se configura con ['es', 'en'] para activar el decodificador en espanol.
#   Fundamento: "Spanish TrOCR" (arXiv 2407.06950, 2024) - investigadores
#   de la UPV demostraron que los motores OCR entrenados en ingles fallan
#   con la N~ y los acentos del espanol. Configurar 'es' mejora la precision.
#
# PREPROCESAMIENTO DE IMAGEN (OpenCV):
#   Antes de leer el texto, mejoramos la imagen con:
#   1. CLAHE (Contrast Limited Adaptive Histogram Equalization):
#      Ecualiza el histograma de forma local (por bloques 8x8).
#      Mejora el contraste en zonas con tinta desvanecida.
#      Fundamento: Reza (2004), tileGridSize=8x8 es el punto optimo.
#   2. Binarizacion Otsu:
#      Convierte la imagen a blanco/negro puro para facilitar la lectura.
#      Umbral calculado automaticamente (metodo estadistico de Otsu).
#
# MEJORAS BASADAS EN ARTICULOS CIENTIFICOS DE ESPANA (2023-2025):
#   1. Language Specific Decoder: ['es','en'] en EasyOCR (Spanish TrOCR, 2024)
#   2. Contrast Tuning: contrast_ths=0.1, adjust_contrast=0.5 (PreP-OCR, 2025)
#   3. Normalizacion NLP + Proteccion de la N~ (KIE Docs Hispanos, IberLEF)
#   4. Correccion Heuristica O/0, I/1, S/5, B/8 (UNED, 2024)

import re
import cv2
import easyocr
import unicodedata
import pandas as pd

# Inicializacion del motor OCR con soporte bilingue espanol + ingles
# gpu=False: usar CPU (no requiere GPU dedicada, mas lento pero portable)
lector_ocr = easyocr.Reader(['es', 'en'], gpu=False)


def normalizar_texto_hispano(texto):
    # Normaliza el texto del OCR para comparacion robusta.
    # PROBLEMA: El OCR puede devolver 'Garcia' con acento o sin el.
    # SOLUCION: Quitar todos los acentos, pero PROTEGER la N~.
    #
    # Paso 1: Proteger la N~ con placeholder '||ENYE||'
    # Paso 2: Aplicar NFD (descompone vocales acentuadas en letra + tilde)
    # Paso 3: Eliminar marcas de acento (categoria Unicode 'Mn')
    # Paso 4: Restaurar la N~ original
    texto = texto.upper().replace('\u00d1', '||ENYE||')
    texto_nfd = unicodedata.normalize('NFD', texto)
    texto_limpio = ''.join([c for c in texto_nfd if unicodedata.category(c) != 'Mn'])
    return texto_limpio.replace('||ENYE||', '\u00d1')


def preprocesar_para_ocr(ruta):
    # Mejora el contraste y binariza la imagen antes del OCR.
    # CLAHE con tileGridSize=(8,8): tamano optimo segun Reza (2004).
    # GaussianBlur(3,3): suaviza el ruido de grano antes de binarizar.
    # Binarizacion Otsu: umbral automatico basado en histograma bimodal.
    img  = cv2.imread(str(ruta))
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gris  = clahe.apply(gris)
    suavizado = cv2.GaussianBlur(gris, (3, 3), 0)
    _, binaria = cv2.threshold(suavizado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binaria


def corregir_confusion_ocr(texto):
    # Genera 3 variantes del texto para corregir confusiones visuales del OCR.
    # Los documentos escaneados a baja resolucion causan que el OCR confunda
    # caracteres de forma similar: O/0, I/1, S/5, B/8.
    # Se generan las variantes y se busca el CURP en cada una.
    # Tecnica avalada en: UNED - LLM Post-processing OCR Espanol (2024)
    variantes = [texto]
    v1 = texto.replace('0', 'O').replace('1', 'I').replace('5', 'S').replace('8', 'B')
    v2 = texto.replace('O', '0').replace('I', '1').replace('S', '5')
    variantes.append(v1)
    variantes.append(v2)
    return variantes


# Patron oficial CURP - Norma SEGOB / RENAPO
# Estructura de 18 caracteres:
#   [4 letras iniciales][6 digitos fecha][H/M sexo]
#   [2 letras entidad][3 consonantes apellidos][1 consonante nombre][1 digito verificador]
# Nota: La N~ nunca aparece en el CURP; si el apellido la tiene, se sustituye por X.
PATRON_CURP = re.compile(
    r'[A-Z]{4}[0-9]{6}[HM][A-Z]{2}[B-DF-HJ-NP-TV-Z]{3}[A-Z0-9][0-9]'
)

# Ontologia de campos obligatorios para Actas de Nacimiento.
# Incluimos variantes tipograficas comunes del OCR (ej: ACT4 por ACTA)
CAMPOS_ACTA = {
    'ACTA'       : ['ACTA', 'ACTA.', 'ACT4'],
    'NACIMIENTO' : ['NACIMIENTO', 'NACI MIENTO', 'NACIM1ENTO'],
    'NOMBRE'     : ['NOMBRE', 'N0MBRE', 'NOMBR'],
    'MUNICIPIO'  : ['MUNICIPIO', 'MUNICI PIO', 'MPIO'],
}


def barrera_3(ruta, clase_documento):
    # Pipeline completo de validacion textual para un documento.
    # 1. Preprocesar imagen (CLAHE + Otsu)
    # 2. Leer texto con EasyOCR
    # 3. Filtrar por confianza >= 30%
    # 4. Normalizar texto (quitar acentos, proteger N~)
    # 5. Validar segun tipo: CURP con Regex, Acta con campos
    # Retorna: (ok, confianza, campos, mensaje)

    img_mejorada = preprocesar_para_ocr(ruta)

    # Leer texto con parametros optimizados para documentos degradados.
    # adjust_contrast=0.5: relee zonas con tinta borrosa (PreP-OCR, 2025)
    resultados = lector_ocr.readtext(
        img_mejorada, detail=1, paragraph=False,
        contrast_ths=0.1, adjust_contrast=0.5
    )

    if not resultados:
        return False, 0.0, [], 'Error: No se detecto texto en el documento.'

    # Filtrar texto con confianza OCR minima del 30%
    textos_confiables = [(det[1], det[2]) for det in resultados if det[2] >= 0.30]
    if not textos_confiables:
        return False, 0.0, [], 'Error: Confianza de lectura insuficiente.'

    confianza_promedio = sum(c for _, c in textos_confiables) / len(textos_confiables)
    texto_completo = normalizar_texto_hispano(
        ' '.join([t for t, _ in textos_confiables])
    )

    if clase_documento == 'curp':
        # Buscar CURP en las 3 variantes para corregir posibles errores del OCR
        curp_encontrado = None
        for variante in corregir_confusion_ocr(texto_completo):
            match = PATRON_CURP.search(variante)
            if match:
                curp_encontrado = match.group(0)
                break
        if curp_encontrado:
            return (True, confianza_promedio, [curp_encontrado],
                    'CURP Validado: ' + curp_encontrado +
                    ' | Precision OCR: ' + str(round(confianza_promedio*100, 1)) + '%')
        return (False, confianza_promedio, [],
                'Error: Estructura CURP no detectada. Confianza: ' +
                str(round(confianza_promedio*100, 1)) + '%')

    elif clase_documento == 'actas_nacimiento':
        hallados  = []
        faltantes = []
        for campo, variantes_campo in CAMPOS_ACTA.items():
            if any(v in texto_completo for v in variantes_campo):
                hallados.append(campo)
            else:
                faltantes.append(campo)
        # Regla de decision: si faltan 2 o mas campos -> RECHAZADO
        if len(faltantes) >= 2:
            return (False, confianza_promedio, hallados,
                    'Error: Faltan metadatos requeridos: ' + str(faltantes))
        return (True, confianza_promedio, hallados,
                'Metadatos validados: ' + str(hallados) +
                ' | Confianza OCR: ' + str(round(confianza_promedio*100, 1)) + '%')

    return False, 0.0, [], 'Clase de documento no soportada: ' + str(clase_documento)


# Procesar solo documentos que aprobaron la Barrera 2
b3_res = []
aprobados_b2 = set(B2[B2['b2_ok']]['archivo']) if 'B2' in globals() else set()

for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    if nombre not in aprobados_b2: continue

    fila_b1 = B1[B1['archivo'] == nombre]
    if fila_b1.empty: continue
    clase_pred = fila_b1.iloc[0]['clase_pred']

    ok, conf_ocr, campos, msg = barrera_3(ruta, clase_pred)
    b3_res.append({
        'archivo'        : nombre,
        'clase_pred'     : clase_pred,
        'confianza_ocr_%': round(conf_ocr * 100, 1),
        'campos_hallados': str(campos),
        'b3_ok'          : ok,
        'mensaje'        : msg
    })

B3 = pd.DataFrame(b3_res)

print('=== BARRERA 3 COMPLETADA ===')
print(f'Procesados: {len(B3)} archivos. Resultados listos para la Tabla Maestra.\n')


In [8]:
# =============================================================
# BARRERA 5: Correspondencia Identitaria Alumno-Documento (PENDIENTE)
# Implementación proyectada: Similitud léxica mediante distancia de Levenshtein contra base de datos
# Referencia metodológica: Shi & Jain (2019), DocFace+ (IEEE TBIOM, DOI:10.1109/TBIOM.2019.2897807)
# =============================================================
print('BARRERA 5 — Correspondencia alumno-documento: PENDIENTE')

BARRERA 5 — Correspondencia alumno-documento: PENDIENTE


In [ ]:
# ==============================================================
# CELDA FINAL: TABLA MAESTRA DE RESULTADOS DEL PIPELINE
# ==============================================================
# Esta celda consolida los resultados de las 3 barreras en una
# tabla maestra unica que muestra el veredicto final por documento.
#
# ESTRUCTURA:
#   - B1: clase predicha, confianza de la CNN, si aprobó
#   - B2: si la imagen tuvo buena calidad
#   - B3: si el contenido textual es valido, confianza OCR
#   - estado_final: ACEPTADO si las 3 son True, RECHAZADO si alguna falla
#   - observaciones: resumen de los mensajes de cada barrera
#
# REGLA DE CASCADA: Un documento rechazado en B1 NO llega a B2 ni B3.
#   Por eso algunos documentos tendran B2 y B3 en estado N/A.
#
# Al terminar, se exporta el resultado como JSON para que la pagina
# web del administrador pueda leerlo y mostrar los resultados.

import pandas as pd
import json

def construir_tabla_maestra(B1, B2, B3):
    # Evitar multiplicación de filas eliminando duplicados por nombre de archivo
    df_b1_raw = B1.drop_duplicates(subset=['archivo'])
    df_b2_raw = B2.drop_duplicates(subset=['archivo']) if not B2.empty else B2
    df_b3_raw = B3.drop_duplicates(subset=['archivo']) if not B3.empty else B3

    # B1: detectar columna de confianza
    col_conf_b1 = next((c for c in ['confianza_b1_%','confianza_%'] if c in df_b1_raw.columns), None)
    cols_b1 = ['archivo', 'clase_pred', 'b1_ok', 'mensaje']
    if col_conf_b1: cols_b1.insert(2, col_conf_b1)
    df_b1 = df_b1_raw[cols_b1].copy()
    rename_b1 = {'mensaje': 'obs_b1'}
    if col_conf_b1: rename_b1[col_conf_b1] = 'conf_b1_%'
    df_b1.rename(columns=rename_b1, inplace=True)
    if 'conf_b1_%' not in df_b1.columns: df_b1['conf_b1_%'] = 'N/D'

    # B2: columna de mensaje
    if not df_b2_raw.empty:
        col_msg_b2 = 'mensaje' if 'mensaje' in df_b2_raw.columns else df_b2_raw.columns[-1]
        df_b2 = df_b2_raw[['archivo', 'b2_ok', col_msg_b2]].copy()
        df_b2.rename(columns={col_msg_b2: 'obs_b2'}, inplace=True)
    else:
        df_b2 = pd.DataFrame(columns=['archivo', 'b2_ok', 'obs_b2'])

    # B3: confianza OCR es opcional segun la version del pipeline
    if not df_b3_raw.empty:
        col_msg_b3  = 'mensaje' if 'mensaje' in df_b3_raw.columns else df_b3_raw.columns[-1]
        col_conf_b3 = next((c for c in ['confianza_ocr_%','confianza_%'] if c in df_b3_raw.columns), None)
        cols_b3 = ['archivo', 'b3_ok', col_msg_b3]
        if col_conf_b3: cols_b3.insert(2, col_conf_b3)
        df_b3 = df_b3_raw[cols_b3].copy()
        rename_b3 = {col_msg_b3: 'obs_b3'}
        if col_conf_b3: rename_b3[col_conf_b3] = 'conf_ocr_%'
        df_b3.rename(columns=rename_b3, inplace=True)
        if 'conf_ocr_%' not in df_b3.columns: df_b3['conf_ocr_%'] = 'N/D'
    else:
        df_b3 = pd.DataFrame(columns=['archivo', 'b3_ok', 'obs_b3', 'conf_ocr_%'])

    # Cruce con left join para conservar todos los documentos
    master = df_b1.merge(df_b2, on='archivo', how='left')
    master = master.merge(df_b3, on='archivo', how='left')
    master['b2_ok']      = master['b2_ok'].fillna('N/A')
    master['b3_ok']      = master['b3_ok'].fillna('N/A')
    master['obs_b2']     = master['obs_b2'].fillna('No procesado (rechazado en B1)')
    master['obs_b3']     = master['obs_b3'].fillna('No procesado (rechazado en B2 o B1)')
    master['conf_ocr_%'] = master['conf_ocr_%'].fillna('N/D')
    return master

def calcular_estado_final(row):
    if row['b1_ok'] == True and row['b2_ok'] == True and row['b3_ok'] == True:
        return 'ACEPTADO'
    return 'RECHAZADO'

def construir_observaciones(row):
    obs = ['B1: ' + str(row['obs_b1'])]
    if 'No procesado' not in str(row['obs_b2']): obs.append('B2: ' + str(row['obs_b2']))
    if 'No procesado' not in str(row['obs_b3']): obs.append('B3: ' + str(row['obs_b3']))
    return ' | '.join(obs)

if 'B1' in globals() and 'B2' in globals() and 'B3' in globals():
    df_master = construir_tabla_maestra(B1, B2, B3)
    df_master['estado_final']  = df_master.apply(calcular_estado_final, axis=1)
    df_master['observaciones'] = df_master.apply(construir_observaciones, axis=1)

    columnas_vista = ['archivo','clase_pred','conf_b1_%','b1_ok','b2_ok','b3_ok','conf_ocr_%','estado_final']

    print('\n' + '='*65)
    print('   TABLA MAESTRA DE RESULTADOS - PIPELINE DE VERIFICACION')
    print('='*65)
    print(df_master[columnas_vista].to_string(index=False))

    print('\n' + '='*65)
    print('   OBSERVACIONES POR DOCUMENTO')
    print('='*65)
    print(df_master[['archivo','observaciones']].to_string(index=False))

    total     = len(df_master)
    aceptados = (df_master['estado_final'] == 'ACEPTADO').sum()
    print('\n' + '='*65)
    print('   RESUMEN ESTADISTICO')
    print('='*65)
    print('  Total procesados : ' + str(total))
    print('  ACEPTADOS        : ' + str(aceptados) + ' (' + str(round(aceptados/total*100,1)) + '%)')
    print('  RECHAZADOS       : ' + str(total-aceptados) + ' (' + str(round((total-aceptados)/total*100,1)) + '%)')

    ruta_json = BASE_DIR / 'resultados_pipeline.json'
    with open(ruta_json, 'w', encoding='utf-8') as f:
        json.dump({
            'resumen': {'total': total, 'aceptados': int(aceptados), 'rechazados': int(total-aceptados)},
            'documentos': df_master.to_dict(orient='records')
        }, f, ensure_ascii=False, indent=2)
    print('\n  JSON exportado en: ' + str(ruta_json))

else:
    if 'B1' in globals(): print('Columnas B1:', list(B1.columns))
    if 'B2' in globals(): print('Columnas B2:', list(B2.columns))
    if 'B3' in globals(): print('Columnas B3:', list(B3.columns))
    print('[ERROR] Ejecuta primero las celdas de las Barreras 1, 2 y 3.')

